## 1. Archivos de Configuración y Compilación (Raíz del Paquete)

A diferencia de los paquetes desarrollados en Python (ament_python), este módulo está desarrollado en **C++**, por lo que requiere un proceso de construcción explícito. Los siguientes archivos definen las reglas de compilación, las dependencias y los metadatos del paquete dentro del ecosistema ROS 2.

---

### **1.1 package.xml** - El Manifiesto del Paquete

Este archivo XML actúa como el "pasaporte" o documento de identidad del paquete `dofbot_telemetry_cpp`. 

**¿Qué hace el código?**
- Define los metadatos esenciales: nombre del paquete, versión, autor y tipo de licencia (MIT).
- Declara explícitamente el sistema de construcción utilizado (`ament_cmake`).
- Enumera las dependencias necesarias para la ejecución (`exec_depend`), exigiendo que las librerías `rclcpp` (cliente de ROS 2 para C++) y `dofbot_interfaces` (mensajes personalizados) estén presentes en el entorno.

**Importancia:** Garantiza que el gestor de dependencias de ROS 2 (como `rosdep`) instale y enlace automáticamente todas las librerías externas necesarias antes de intentar compilar el código fuente, evitando errores de dependencias faltantes.

---

### **1.2 CMakeLists.txt** - Reglas de Construcción

Es el plano arquitectónico para el compilador. Orquesta cómo el código fuente en `.cpp` se transforma en nodos ejecutables dentro de ROS 2.

**¿Qué hace el código?**
- `find_package(...)`: Localiza las librerías requeridas en el sistema (`ament_cmake`, `rclcpp`, `dofbot_interfaces`).
- `add_executable(...)`: Declara los dos programas principales a compilar. Aquí se establece que el archivo `src/telemetry.cpp` generará el ejecutable **`telemetry_node_pub`**, y `src/telemetry_subs.cpp` generará **`telemetry_node_sub`**.
- `ament_target_dependencies(...)`: Vincula las librerías de ROS 2 a cada ejecutable para que reconozcan las funciones del framework.
- `install(TARGETS ...)`: Define la ruta destino donde se guardarán los binarios compilados (`lib/${PROJECT_NAME}`) para que el comando `ros2 run` pueda encontrarlos fácilmente.

**Importancia:** Estandariza el proceso de compilación para que sea reproducible en cualquier máquina. Además, incorpora banderas de advertencia (`-Wall -Wextra`) que obligan a mantener un código C++ limpio y libre de malas prácticas de memoria.

---

### **1.3 LICENSE.txt** - Términos de Distribución

Contiene los términos legales bajo los cuales se distribuye el software de este paquete.

**¿Qué hace el código?**
- Implementa la **Licencia MIT** (Open Source).

**Importancia:** En el desarrollo colaborativo y la robótica, definir una licencia clara permite que otros equipos, investigadores o desarrolladores puedan utilizar, modificar y escalar la arquitectura de telemetría del Dofbot de forma legal y segura, fomentando la innovación compartida.

## 2. Análisis del Código Fuente (`src/`)

En esta sección se analiza a nivel de líneas de código el comportamiento de los dos nodos principales que gestionan la telemetría del robot DOFBot en C++. Se destaca el uso de la programación orientada a objetos (POO) mediante la herencia de la clase `rclcpp::Node`.

---

### **2.1 telemetry.cpp** - Implementación del Nodo Publicador (`telemetry_node`)

Este archivo se encarga de instanciar un nodo emisor que empaqueta los datos de coordenadas y estado del robot para transmitirlos de manera periódica al ecosistema de ROS 2.

#### **Desglose Técnico del Código:**

* **Líneas de Inclusión (`#include`):** Se vincula `"dofbot_interfaces/msg/telemetry.hpp"`, que contiene la definición de la estructura del mensaje personalizado, y `"rclcpp/rclcpp.hpp"`, el núcleo de ROS 2 para C++. Se añade `using namespace std::chrono_literals;` para poder expresar unidades de tiempo de forma natural (como `1s`).
* **Definición de la Clase (`TelemetryPub`):**
    Hereda de de forma pública de `rclcpp::Node`. El constructor inicializa el nodo con el nombre oficial de **`telemetry_node`**.
* **Configuración del Publicador (`create_publisher`):**
    Se inicializa el objeto `publisher_` apuntando al tópico **`/telemetry_cpp`** con un canal de comunicación de tipo `dofbot_interfaces::msg::Telemetry` y una profundidad de cola (QoS) de `10`.
* **Temporizador de Pared (`create_wall_timer`):**
    Se crea un *timer* configurado a `1s` (frecuencia de 1 Hz) emparejado mediante `std::bind` con la función miembro `callback_telemetry`. Esto automatiza la ejecución cíclica del nodo.
* **Función de Retroalimentación (`callback_telemetry`):**
    Cada segundo, este método crea una instancia local del mensaje (`telemetry_msg`), llena sus campos con valores de seguridad por defecto (`status = "STAND BY"`, y posiciones `pos_x`, `pos_y`, `pos_z` en `0.0`) e invoca el método `publish()` para colocar el paquete en la red de ROS 2.
* **Función Principal (`main`):**
    Inicializa el middleware (`rclcpp::init`), crea el puntero compartido del nodo (`std::make_shared`), lo mantiene vivo escuchando eventos del sistema mediante `rclcpp::spin`, y libera los recursos de memoria con `rclcpp::shutdown` al interrumpir la ejecución (`Ctrl+C`).

**Importancia:** Establece la base confiable de un hilo de ejecución dedicado exclusivamente a la transmisión de datos espaciales. Al ejecutarse a 1 Hz en un ciclo síncrono controlado por hardware, garantiza un flujo predecible de telemetría sin saturar el ancho de banda del procesador embebido del DOFBot.

---

### **2.2 telemetry_subs.cpp** - Implementación del Nodo Suscriptor (`telemetry_sub_node`)

Este archivo define la lógica del nodo receptor encargado de capturar las señales de telemetría emitidas en la red del robot para su procesamiento y visualización en consola.

#### **Desglose Técnico del Código:**

* **Gestión de Parámetros (`using std::placeholders::_1;`):**
    Línea crucial en C++ para ROS 2. Reserva el espacio de memoria para el primer argumento que se pasará a la función de callback de manera asíncrona (el mensaje recibido).
* **Definición de la Clase (`TelemetrySub`):**
    Al igual que el publicador, hereda de `rclcpp::Node`. Su constructor bautiza al nodo en la red como **`telemetry_sub_node`**.
* **Configuración de la Suscripción (`create_subscription`):**
    Establece la sintonización del tópico **`/telemetry_cpp`**. Mantiene la misma configuración de tipo de mensaje y tamaño de cola de entrada (`10`). Se vincula directamente a la función privada `on_telemetry_clbk` mediante `std::bind`, pasando el marcador de posición `_1`.
* **Función Callback del Suscriptor (`on_telemetry_clbk`):**
    Es un método constante (`const`) que recibe una referencia de solo lectura (`const &msg`) para optimizar el rendimiento y evitar copias innecesarias en memoria. Utiliza la macro de logging **`RCLCPP_INFO`** para extraer la cadena de texto de estilo C (`msg.status.c_str()`) y las coordenadas de tipo flotante, desplegándolas inmediatamente en la terminal con formato estructurado.

**Importancia:** Permite el monitoreo no bloqueante del estado del robot. Al operar mediante funciones *callback* asíncronas, el nodo permanece dormido y solo consume ciclos de reloj del procesador en el microsegundo exacto en el que ingresa un paquete nuevo al buffer, demostrando la alta eficiencia de rendimiento que ofrece C++ en sistemas robóticos de tiempo real.